In [3]:
import pandas as pd
import numpy as np

# 1. LOAD DATA & DATA INTEGRITY AUDIT
fact = pd.read_csv("../data/raw/Fact_Financials.csv")
dim_acc = pd.read_csv("../data/raw/Dim_Account.csv")
dim_dept = pd.read_csv("../data/raw/Dim_Department.csv")

# Merge into a single analytical dataframe
df = fact.merge(dim_acc, on="Account_ID", how="left").merge(
    dim_dept, on="Department_ID", how="left"
)
df["Date"] = pd.to_datetime(df["Date"])

print("--- 1. DATA INTEGRITY AUDIT ---")
print(f"Missing values across entire dataset:\n{df.isnull().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(
    f"Unique Accounts: {df['Account_ID'].nunique()} | Unique Departments: {df['Department_ID'].nunique()}\n"
)

--- 1. DATA INTEGRITY AUDIT ---
Missing values across entire dataset:
Date                 0
Account_ID           0
Department_ID        0
Scenario             0
Amount               0
Statement_Section    0
Category             0
Account_Name         0
Department_Name      0
Cost_Center_Owner    0
dtype: int64
Duplicate rows: 0
Unique Accounts: 14 | Unique Departments: 5



In [4]:
# 2. OPEX & COGS VARIANCE ANALYSIS (ACTUAL vs BUDGET)
# Pivot Scenario to calculate Variance = Actual - Budget
pivot_df = df.pivot_table(
    index=[
        "Date",
        "Statement_Section",
        "Department_Name",
        "Account_Name",
        "Account_ID",
    ],
    columns="Scenario",
    values="Amount",
).reset_index()

pivot_df["Variance_$"] = pivot_df["Actual"] - pivot_df["Budget"]
pivot_df["Variance_%"] = (pivot_df["Variance_$"] / pivot_df["Budget"]) * 100

print("--- 2. TOP 5 UNFAVORABLE OPEX & COGS VARIANCES (OVERSPEND) ---")
opex_cogs = pivot_df[pivot_df["Statement_Section"].isin(["OPEX", "COGS"])]
top_overspend = (
    opex_cogs.groupby(["Department_Name", "Account_Name"])
    .agg(
        Total_Budget=("Budget", "sum"),
        Total_Actual=("Actual", "sum"),
        Total_Variance=("Variance_$", "sum"),
        Avg_Variance_Pct=("Variance_%", "mean"),
    )
    .sort_values(by="Total_Variance", ascending=False)
    .head(5)
)

# Format helper for clean console printing
format_fn = lambda x: (
    f"${x:,.2f}"
    if isinstance(x, (int, float)) and abs(x) > 100
    else f"{x:.2f}%" if isinstance(x, float) else x
)
print(
    top_overspend.map(format_fn)
    if hasattr(top_overspend, "map")
    else top_overspend.applymap(format_fn)
)

--- 2. TOP 5 UNFAVORABLE OPEX & COGS VARIANCES (OVERSPEND) ---
                                                                   Total_Budget   
Department_Name          Account_Name                                             
Research & Development   R&D Cloud Testing & Staging Compute        $563,760.00  \
Marketing                Performance Advertising & Demand Gen     $1,064,880.00   
General & Administrative Legal, Audit & Professional Fees           $375,840.00   
Commercial Sales         Salaries & Benefits - Sales & Marketing  $2,255,040.00   
General & Administrative Salaries & Benefits - G&A                $1,190,160.00   

                                                                   Total_Actual   
Department_Name          Account_Name                                             
Research & Development   R&D Cloud Testing & Staging Compute        $667,579.41  \
Marketing                Performance Advertising & Demand Gen     $1,168,095.86   
General & Administrativ

In [5]:
# 3. REVENUE VARIANCE ANALYSIS (UNDERPERFORMANCE)
print("\n--- 3. TOP REVENUE MISSES (UNDERPERFORMANCE) ---")
rev_df = pivot_df[pivot_df["Statement_Section"] == "Revenue"]
top_rev_miss = (
    rev_df.groupby(["Department_Name", "Account_Name"])
    .agg(
        Total_Budget=("Budget", "sum"),
        Total_Actual=("Actual", "sum"),
        Total_Variance=("Variance_$", "sum"),
        Avg_Variance_Pct=("Variance_%", "mean"),
    )
    .sort_values(by="Total_Variance", ascending=True)
)
print(
    top_rev_miss.map(format_fn)
    if hasattr(top_rev_miss, "map")
    else top_rev_miss.applymap(format_fn)
)


--- 3. TOP REVENUE MISSES (UNDERPERFORMANCE) ---
                                                             Total_Budget   
Department_Name       Account_Name                                          
Professional Services Implementation & Consulting Services  $1,864,800.00  \
Commercial Sales      Mid-Market SaaS Subscription          $3,729,600.00   
                      Enterprise SaaS Subscription          $7,326,000.00   

                                                             Total_Actual   
Department_Name       Account_Name                                          
Professional Services Implementation & Consulting Services  $1,751,536.81  \
Commercial Sales      Mid-Market SaaS Subscription          $3,700,849.36   
                      Enterprise SaaS Subscription          $7,365,589.85   

                                                           Total_Variance   
Department_Name       Account_Name                                          
Professional Services Im

In [6]:
# 4. STATISTICAL OUTLIER DETECTION (Z-Score on Monthly Variances)
# Identifying shock events rather than gradual trends
pivot_df["Variance_ZScore"] = pivot_df.groupby("Account_ID")["Variance_$"].transform(
    lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) != 0 else 1)
)

outliers = pivot_df[pivot_df["Variance_ZScore"].abs() > 2.2][
    [
        "Date",
        "Department_Name",
        "Account_Name",
        "Actual",
        "Budget",
        "Variance_$",
        "Variance_ZScore",
    ]
]
print("\n--- 4. STATISTICAL ANOMALIES (Z-Score > 2.2) ---")
print(outliers.to_string(index=False))


--- 4. STATISTICAL ANOMALIES (Z-Score > 2.2) ---
      Date          Department_Name                     Account_Name    Actual   Budget  Variance_$  Variance_ZScore
2025-06-01         Commercial Sales     Enterprise SaaS Subscription 676408.37 605000.0    71408.37         2.255548
2025-08-01 General & Administrative Legal, Audit & Professional Fees  73680.00  31680.0    42000.00         3.309694


In [7]:
# 5. MARKETING SPEND VS SUBSCRIPTION REVENUE (EFFICIENCY CHECK)
# Check 1-month and 2-month lagged correlation
mkt_spend = df[
    (df["Account_ID"] == "OPX_201") & (df["Scenario"] == "Actual")
].set_index("Date")["Amount"]
saas_rev = df[(df["Account_ID"] == "REV_101") & (df["Scenario"] == "Actual")].set_index(
    "Date"
)["Amount"]

mkt_analysis = pd.DataFrame({"Marketing_Spend": mkt_spend, "SaaS_Revenue": saas_rev})
mkt_analysis["Rev_Lag_1M"] = mkt_analysis["SaaS_Revenue"].shift(-1)
mkt_analysis["Rev_Lag_2M"] = mkt_analysis["SaaS_Revenue"].shift(-2)

corr_0m = mkt_analysis["Marketing_Spend"].corr(mkt_analysis["SaaS_Revenue"])
corr_1m = mkt_analysis["Marketing_Spend"].corr(mkt_analysis["Rev_Lag_1M"])

print("\n--- 5. MARKETING SPEND TO REVENUE CORRELATION ---")
print(f"Same-Month Correlation: {corr_0m:.2f}")
print(f"1-Month Lagged Correlation: {corr_1m:.2f}")


--- 5. MARKETING SPEND TO REVENUE CORRELATION ---
Same-Month Correlation: -0.51
1-Month Lagged Correlation: -0.04
